# SPARQL query tests

Run the cells in order or use **Run All**. The Turtle graph is loaded once, without a reasoner. Queries are read from `rdf/queries.sparql`; rerun the cells after modifying that file. The first seven checks should return zero rows. The remaining searches may return no results for this sample.


If RDFLib or pandas is not available in the kernel, run `%pip install rdflib pandas` in a cell, then restart the kernel if necessary.


In [14]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display
from rdflib import Graph

# Works from transformation/, Group07, or the repository root.
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "HandsOn" / "Group07",
]
group_dir = next(
    (path for path in candidates if (path / "rdf" / "queries.sparql").is_file()),
    None,
)
if group_dir is None:
    raise FileNotFoundError("Open the notebook from transformation/, Group07, or the repository root.")

graph = Graph()
graph.parse(group_dir / "rdf" / "knowledge-graph.ttl", format="turtle")
print(f"Graph loaded: {len(graph):,} triples")

source = (group_dir / "rdf" / "queries.sparql").read_text(encoding="utf-8-sig")
# Split this file into blocks containing one SELECT query each.
parts = re.split(r"(?m)^SELECT\b", source)
prefixes = "\n".join(
    line for line in parts[0].splitlines()
    if line.strip().upper().startswith("PREFIX ")
)
queries = [prefixes + "\nSELECT" + part for part in parts[1:]]
if len(queries) != 12:
    raise ValueError(f"Expected 12 queries, found {len(queries)}. Update the cells if the list changes.")
print(f"{len(queries)} queries available")


def run_query(index, expected_empty=False):
    result = graph.query(queries[index - 1])
    rows = list(result)
    table = pd.DataFrame(
        [[None if value is None else str(value) for value in row] for row in rows],
        columns=[str(variable) for variable in result.vars],
    )
    if expected_empty:
        print("OK: no anomaly detected." if not rows else f"Check: {len(rows)} result(s).")
    else:
        print(f"{len(rows)} result(s).")
    display(table.head(50))
    if len(table) > 50:
        print("Display limited to the first 50 rows; all results remain available in the DataFrame.")
    return table


Graph loaded: 16,093 triples
12 queries available


## 1. TrainStop without a station

Expected result: zero rows.


In [15]:
results_1 = run_query(1, expected_empty=True)


OK: no anomaly detected.


,stop


## 2. TrainStop with more than one station

Expected result: zero rows.


In [16]:
results_2 = run_query(2, expected_empty=True)


OK: no anomaly detected.


,stop,n


## 3. Station without exactly one EVA number

Expected result: zero rows.


In [17]:
results_3 = run_query(3, expected_empty=True)


OK: no anomaly detected.


,st,n


## 4. Several Station individuals sharing one EVA number

Expected result: zero rows.


In [18]:
results_4 = run_query(4, expected_empty=True)


OK: no anomaly detected.


,eva,n


## 5. Broken Station -> City -> State chain

Expected result: zero rows.


In [19]:
results_5 = run_query(5, expected_empty=True)


OK: no anomaly detected.


,st,city


## 6. City located in more than one State

Expected result: zero rows.


In [20]:
results_6 = run_query(6, expected_empty=True)


OK: no anomaly detected.


,city,n


## 7. Domain/range violations (run WITHOUT a reasoner)

Expected result: zero rows.


In [21]:
results_7 = run_query(7, expected_empty=True)


OK: no anomaly detected.


,s,o


## 8. Instance counts per class

An empty result may depend on the sample data.


In [22]:
results_8 = run_query(8, expected_empty=False)


0 result(s).


,class,n


## 9. Stops at a given station (adjust EVA), with delays

An empty result may depend on the sample data.


In [23]:
results_9 = run_query(9, expected_empty=False)


0 result(s).


,stop,line,plannedArrival,arrivalDelayMinutes


## 10. Stops whose path mentions a given previous station

An empty result may depend on the sample data.


In [24]:
results_10 = run_query(10, expected_empty=False)


0 result(s).


,stop,path


## 11. Stations in a bounding box (around Berlin)

An empty result may depend on the sample data.


In [25]:
results_11 = run_query(11, expected_empty=False)


0 result(s).


,name,lat,long


## 12. Average arrival delay per state

An empty result may depend on the sample data.


In [26]:
results_12 = run_query(12, expected_empty=False)


0 result(s).


,stateName,avgDelay,stops
